<a href="https://colab.research.google.com/github/fourmodern/2026_aidrugdiscovery/blob/main/Day06_LLM_Agent/t050_simple-local-rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 로컬 RAG 파이프라인 처음부터 만들기 — PDE5 저해제 문헌 편

## 출처 및 저작자 표기

이 노트북은 **Daniel Bourke (mrdbourke)** 가 만든 오픈소스 튜토리얼
[`mrdbourke/simple-local-rag`](https://github.com/mrdbourke/simple-local-rag) 의
원본 노트북 [`00-simple-local-rag.ipynb`](https://github.com/mrdbourke/simple-local-rag/blob/main/00-simple-local-rag.ipynb)
을 **기반(derivative)** 으로 하며, 본 신약개발 강의용으로 개편한 것입니다.

| 항목 | 내용 |
|---|---|
| 원저자 | Daniel Bourke ([@mrdbourke](https://github.com/mrdbourke)) |
| 원본 저장소 | `https://github.com/mrdbourke/simple-local-rag` |
| 원본 노트북 | `00-simple-local-rag.ipynb` |
| 원본 코퍼스 | *Human Nutrition* 오픈 교과서 PDF (1,208쪽) |
| 본 노트북 코퍼스 | **PDE5 저해제 관련 오픈액세스 논문 6편** (Europe PMC) |

### 개편 내용

1. **코퍼스 교체** — 영양학 교과서 → PDE5 저해제 오픈액세스 논문 6편
2. **질의 교체** — 영양학 질문 → PDE5 약리·안전성·리포지셔닝 질문
3. **분량 압축** — 원본의 추론 최적화(flash-attention, TensorRT-LLM 등) 곁가지를
   제거하고 RAG 5단계 핵심만 남김 (제거한 주제는 맨 끝 "더 알아보기" 참고)
4. **출처 추적 추가** — 청크마다 PMCID·논문 제목·쪽 번호를 메타데이터로 붙여
   답변의 근거를 되짚을 수 있게 함

### 라이선스 (확인 결과)

> **원본 저장소에는 LICENSE 파일이 없습니다.**
>
> 2026-09-01 기준 확인:
> - GitHub API (`https://api.github.com/repos/mrdbourke/simple-local-rag`) 의
>   `license` 필드 → **`null`**
> - `LICENSE`, `LICENSE.md`, `LICENSE.txt`, `COPYING` raw 파일 → 모두 **404**
> - 저장소 루트 파일 목록에도 라이선스 파일 없음
>
> 즉 **명시된 오픈소스 라이선스가 없어, 기본적으로 원저자가 모든 저작권을 보유**합니다.
> 본 노트북은 **비영리 교육 목적의 참고·개편물**로만 사용하며, 재배포·상업적 이용 전에는
> **원저자에게 직접 허락을 받아야 합니다**. (라이선스명을 임의로 추정해 기재하지 않았습니다.)

그 밖에 사용하는 구성요소의 라이선스는 각 프로젝트를 따릅니다 —
PyMuPDF(AGPL-3.0), sentence-transformers(Apache-2.0), Qwen3(Apache-2.0),
Europe PMC 논문(각 논문의 CC 라이선스).

## 0. 실행 환경 / 요구사항

- **GPU 권장** (Colab: 런타임 > 런타임 유형 변경 > T4 GPU). **CPU에서도 끝까지 동작**하지만
  임베딩과 답변 생성이 느립니다(CPU 기준 답변 1건에 1~3분).
- 내려받는 것
  - **PDE5 논문 PDF 6편** (Europe PMC 오픈액세스, 합계 약 20MB) — 코퍼스
  - `sentence-transformers/all-mpnet-base-v2` (약 420MB, **768차원**) — 임베딩
  - `Qwen/Qwen3-1.7B` (약 3.4GB) — 생성 LLM. **gated 아님, HF 로그인 불필요**
- `google/gemma-2b-it` 를 쓰려면 HF에서 라이선스 동의 + 토큰 로그인이 필요합니다
  (아래 `USE_GATED_GEMMA = True` 로 전환).

### 우리가 만들 것 (RAG 5단계)

| 단계 | 하는 일 | 도구 |
|---|---|---|
| 1 | PDF에서 텍스트 추출 | PyMuPDF (`fitz`) |
| 2 | 문장 분리 → **10문장 청크** | spaCy `sentencizer` |
| 3 | 청크 → **768차원 임베딩** → CSV 저장 | `all-mpnet-base-v2` |
| 4 | 질의 임베딩 → **dot product** top-k 검색 | `util.dot_score` |
| 5 | 검색 문맥 + 질의 → 프롬프트 → **LLM 생성** | `apply_chat_template` + Qwen3 |

**RAG(Retrieval Augmented Generation)** 의 핵심은 *Retrieval*(검색)입니다.
LLM에게 "네가 아는 걸 말해봐"가 아니라 **"이 문서에서 찾은 이 문단들만 근거로 답해"** 라고
시키는 것이라, 답변의 출처를 되짚을 수 있고 환각(hallucination)이 줄어듭니다.


### 핵심 용어

| 용어 | 뜻 |
|---|---|
| **Token** | 모델이 텍스트를 다루는 최소 단위. 영어는 대략 4글자 ≈ 1토큰 |
| **Embedding** | 텍스트를 의미가 담긴 숫자 벡터로 바꾼 것. 여기서는 768차원 |
| **Chunk** | 검색·문맥 주입의 단위로 자른 텍스트 조각. 여기서는 10문장 |
| **Similarity** | 두 벡터가 얼마나 가까운지. 정규화된 벡터에선 내적 = 코사인 유사도 |
| **top-k** | 유사도 상위 k개 청크. 이것이 LLM에 넣을 문맥이 됨 |
| **Retrieval** | 질의로 관련 청크를 찾는 단계 (RAG의 **R**) |
| **Augmentation** | 찾은 청크를 프롬프트에 끼워 넣는 단계 (RAG의 **A**) |
| **Generation** | 증강된 프롬프트로 LLM이 답을 쓰는 단계 (RAG의 **G**) |
| **Hallucination** | 모델이 근거 없이 그럴듯하게 지어내는 것. RAG가 줄이려는 대상 |
| **gated model** | HF에서 라이선스 동의·토큰이 있어야 받는 모델 (예: Gemma) |


In [ ]:
# 실행 환경 확인 (Colab: 런타임 > 런타임 유형 변경 > 하드웨어 가속기 > T4 GPU)
import torch

print("PyTorch:", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("[안내] CPU 런타임입니다. 실행은 되지만 느립니다.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

In [ ]:
# 필요한 패키지 설치
# torch는 Colab에 이미 설치되어 있어 재설치하지 않습니다 (런타임 CUDA 빌드와 어긋날 수 있음)
!pip install -q PyMuPDF                # PDF 텍스트 추출
!pip install -q tqdm                   # 진행률 표시
!pip install -q sentence-transformers  # 임베딩 모델
!pip install -q accelerate             # 대형 모델 로딩
!pip install -q spacy                  # 문장 분리(sentencizer)

# bitsandbytes(4bit 양자화)는 NVIDIA GPU 전용입니다. CPU 런타임에서는 건너뜁니다.
import torch, subprocess
if torch.cuda.is_available():
    subprocess.run("pip install -q bitsandbytes".split())
else:
    print("[안내] CPU 런타임: bitsandbytes 설치를 건너뜁니다.")

---

## 1단계. PDE5 코퍼스 내려받기

코퍼스는 **PDE5(phosphodiesterase type 5) 저해제** 관련 오픈액세스 논문 6편입니다.
모두 **Europe PMC** 에서 `?pdf=render` 로 받는 공개 PDF입니다.

| PMCID | 제목 | 저널 | 연도 |
|---|---|---|---|
| PMC12841899 | Repurposing PDE5-Inhibitors: Sildenafil Drives Arteriogenesis | Int J Mol Sci | 2026 |
| PMC13024863 | Sildenafil Promotes Angiogenesis | Curr Issues Mol Biol | 2026 |
| PMC13076431 | Vardenafil in management of erectile dysfunction | World J Urol | 2026 |
| PMC13149040 | Proarrhythmic Risk of Sildenafil | Biomol Ther | 2026 |
| PMC13111112 | PDE5 inhibitors for cerebral small vessel disease | Front Neurol | 2026 |
| PMC13202541 | Acute sildenafil & arrhythmia susceptibility | J Mol Cell Cardiol Plus | 2026 |

도메인이 셋으로 갈리도록 일부러 골랐습니다 — **리포지셔닝**(혈관신생·측부순환),
**적응증 본류**(발기부전), **심혈관 안전성**(부정맥). 뒤에서 질의를 바꿔가며
검색이 서로 다른 논문을 집어오는지 확인합니다.

> **네트워크가 막히면?** 아래 셀은 실패해도 죽지 않게 `try/except` 로 감쌌습니다.
> **3편 이상만 받아지면 그대로 진행**합니다.

In [ ]:
# PDE5 오픈액세스 논문 PDF 내려받기 (Europe PMC)
import os
import requests

# 코퍼스 정의: 청크마다 이 메타데이터를 붙여 답변의 출처를 추적합니다.
PDE5_CORPUS = [
    {"pmcid": "PMC12841899",
     "title": "Repurposing PDE5-Inhibitors: Sildenafil Drives Arteriogenesis",
     "journal": "Int J Mol Sci", "year": 2026},
    {"pmcid": "PMC13024863",
     "title": "Sildenafil Promotes Angiogenesis",
     "journal": "Curr Issues Mol Biol", "year": 2026},
    {"pmcid": "PMC13076431",
     "title": "Vardenafil in management of erectile dysfunction",
     "journal": "World J Urol", "year": 2026},
    {"pmcid": "PMC13149040",
     "title": "Proarrhythmic Risk of Sildenafil",
     "journal": "Biomol Ther", "year": 2026},
    {"pmcid": "PMC13111112",
     "title": "PDE5 inhibitors for cerebral small vessel disease",
     "journal": "Front Neurol", "year": 2026},
    {"pmcid": "PMC13202541",
     "title": "Acute sildenafil & arrhythmia susceptibility",
     "journal": "J Mol Cell Cardiol Plus", "year": 2026},
]

PDF_DIR = "pde5_pdfs"
os.makedirs(PDF_DIR, exist_ok=True)
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; pde5-rag-lecture/1.0)"}

available_docs = []   # 실제로 받아진 논문만 코퍼스에 포함
for doc in PDE5_CORPUS:
    path = os.path.join(PDF_DIR, f"{doc['pmcid']}.pdf")
    url = f"https://europepmc.org/articles/{doc['pmcid']}?pdf=render"

    if os.path.exists(path) and os.path.getsize(path) > 10_000:
        print(f"[skip] {doc['pmcid']} 이미 있음 ({os.path.getsize(path)/1e6:.1f} MB)")
        available_docs.append({**doc, "path": path})
        continue

    try:
        r = requests.get(url, headers=HEADERS, timeout=90)
        r.raise_for_status()
        # 정말 PDF인지 확인 (차단 시 HTML 오류 페이지가 200으로 오는 경우가 있음)
        if not r.content.startswith(b"%PDF"):
            raise ValueError("응답이 PDF가 아닙니다 (차단되었을 수 있음)")
        with open(path, "wb") as f:
            f.write(r.content)
        print(f"[ok]   {doc['pmcid']} 내려받음 ({len(r.content)/1e6:.1f} MB) — {doc['title'][:45]}")
        available_docs.append({**doc, "path": path})
    except Exception as e:
        print(f"[fail] {doc['pmcid']} 실패: {type(e).__name__}: {e}")

print(f"\n내려받기 성공: {len(available_docs)} / {len(PDE5_CORPUS)} 편")

if len(available_docs) == 0:
    raise RuntimeError(
        "PDF를 한 편도 받지 못했습니다.\n"
        "  - 네트워크/프록시가 europepmc.org 를 막고 있는지 확인하세요.\n"
        "  - 또는 PDF 6개를 수동으로 내려받아 ./pde5_pdfs/ 에 <PMCID>.pdf 로 저장한 뒤\n"
        "    이 셀을 다시 실행하세요."
    )
elif len(available_docs) < 3:
    print("[경고] 3편 미만입니다. 검색 품질이 낮을 수 있으나 실습은 계속 진행됩니다.")

### PDF → 텍스트 (PyMuPDF)

`PyMuPDF`(`fitz`)로 **쪽 단위**로 텍스트를 뽑습니다. 원본 튜토리얼과 달라진 점:

- **여러 편의 PDF**를 한 번에 읽고, 쪽마다 **어느 논문의 몇 쪽인지**를 같이 저장합니다.
- 논문 PDF에는 줄바꿈·**제로폭 문자**(zero-width space)가 섞여 있어
  `https://doi.org` 가 `h​t​t​p​s​:​/​/...` 처럼 깨집니다. 이를 제거하는 정리 단계를 넣었습니다.
- 쪽 번호는 원본처럼 오프셋을 빼지 않고 **PDF 실제 쪽(1부터)** 을 씁니다.

In [ ]:
# PyMuPDF 임포트 (최신 버전은 pymupdf, 구버전은 fitz)
try:
    import pymupdf as fitz
except ImportError:
    import fitz

import re
from tqdm.auto import tqdm

# 논문 PDF에 흔한 제로폭/보이지 않는 문자들 (DOI 링크를 깨뜨리는 주범)
_ZERO_WIDTH = dict.fromkeys(map(ord, "\u200b\u200c\u200d\u2060\ufeff\u00ad"), None)

def text_formatter(text: str) -> str:
    """줄바꿈·제로폭 문자·중복 공백을 정리합니다."""
    text = text.translate(_ZERO_WIDTH)
    return re.sub(r"\s+", " ", text).strip()

def open_and_read_pdfs(docs: list[dict]) -> list[dict]:
    """
    여러 PDF를 쪽 단위로 읽어, 각 쪽의 텍스트와 출처 메타데이터를 함께 반환합니다.

    Returns:
        list[dict]: pmcid / title / journal / year / page_number / 통계 / text
    """
    pages_and_texts = []
    for doc in tqdm(docs, desc="PDF 읽는 중"):
        pdf = fitz.open(doc["path"])
        for page_number, page in enumerate(pdf):
            text = text_formatter(page.get_text())
            pages_and_texts.append({
                "pmcid": doc["pmcid"],
                "title": doc["title"],
                "journal": doc["journal"],
                "year": doc["year"],
                "page_number": page_number + 1,          # PDF 실제 쪽 (1부터)
                "page_char_count": len(text),
                "page_word_count": len(text.split(" ")),
                "page_sentence_count_raw": len(text.split(". ")),
                "page_token_count": len(text) / 4,        # 1 token ~= 4 chars
                "text": text,
            })
        pdf.close()
    return pages_and_texts

pages_and_texts = open_and_read_pdfs(available_docs)
print(f"총 {len(pages_and_texts)} 쪽 추출")
pages_and_texts[0]

In [ ]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
print("논문별 쪽 수 / 단어 수")
display(df.groupby("pmcid").agg(pages=("page_number", "count"),
                                words=("page_word_count", "sum")))
print("\n쪽 단위 통계")
df[["page_char_count", "page_word_count", "page_token_count"]].describe().round(2)

### 문장 분리 (spaCy sentencizer)

`". "` 로 자르면 `Fig. 3` 이나 `et al.` 에서 잘못 끊깁니다.
spaCy의 규칙 기반 `sentencizer` 를 씁니다 — **통계 모델(`en_core_web_sm`) 다운로드가 필요 없어**
가볍고, `English()` 빈 파이프라인에 규칙만 붙이면 됩니다.

In [ ]:
from spacy.lang.en import English

nlp = English()
nlp.add_pipe("sentencizer")

# 동작 확인
doc = nlp("This is a sentence. This another sentence.")
assert len(list(doc.sents)) == 2
print(list(doc.sents))

In [ ]:
# 각 쪽을 문장 리스트로 변환
for item in tqdm(pages_and_texts, desc="문장 분리"):
    item["sentences"] = [str(s) for s in nlp(item["text"]).sents]
    item["page_sentence_count_spacy"] = len(item["sentences"])

df = pd.DataFrame(pages_and_texts)
print("raw('. ' 분리) vs spaCy 문장 수 비교")
df[["page_sentence_count_raw", "page_sentence_count_spacy"]].describe().round(2)

### 10문장 단위 청킹

**왜 청크로 나누나?**

1. 임베딩 모델의 입력 길이가 제한적입니다(`all-mpnet-base-v2` 는 384 토큰에서 잘림).
   논문 한 쪽을 통째로 넣으면 뒷부분이 버려집니다.
2. 검색 단위가 곧 **LLM에 넣을 문맥 단위**입니다. 너무 크면 관련 없는 내용이 섞이고,
   너무 작으면 문맥이 끊깁니다.

여기서는 원본 튜토리얼과 같이 **10문장**을 한 청크로 묶습니다.
(정답은 없습니다 — 실제로는 청크 크기를 바꿔가며 검색 품질을 평가합니다.)

In [ ]:
# 10문장씩 묶어 청크 생성
num_sentence_chunk_size = 10

def split_list(input_list: list, slice_size: int) -> list[list[str]]:
    """리스트를 slice_size 크기의 하위 리스트로 나눕니다. (17개 -> [10, 7])"""
    return [input_list[i:i + slice_size] for i in range(0, len(input_list), slice_size)]

for item in tqdm(pages_and_texts, desc="청킹"):
    item["sentence_chunks"] = split_list(item["sentences"], num_sentence_chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])

print("쪽당 평균 청크 수:", round(pd.DataFrame(pages_and_texts)["num_chunks"].mean(), 2))

### 청크를 개별 항목으로 펼치기 + 노이즈 제거

청크 하나 = 검색 결과 하나가 되도록 평탄화하면서, **출처 메타데이터를 그대로 물려줍니다**.

논문 PDF 특유의 노이즈 두 가지를 걸러냅니다:

- **너무 짧은 청크** (30 토큰 미만) — 머리말/꼬리말/쪽번호 조각
- **참고문헌 목록** — `Kliesch S, Cremers J-F, Krallmann C, ...` 처럼 저자 이니셜이 빽빽한 덩어리.
  내용은 없는데 PDE5 키워드가 잔뜩 들어 있어 **검색 상위를 차지해 버립니다**.
  저자 이니셜 패턴의 **밀도**로 판별합니다(본문은 100단어당 2회 미만, 참고문헌은 18회 이상).

In [ ]:
# 저자 이니셜 패턴: "Kliesch S," "Cremers J-F," "Corona G (2022)" 등
_AUTHOR_INITIALS = re.compile(r"\b[A-Z][a-z]+ [A-Z]{1,3}\b(?=[,;.)\s])")

def looks_like_reference_list(chunk: str, threshold: float = 5.0) -> bool:
    """
    참고문헌 목록으로 보이면 True.
    본문은 100단어당 이니셜 패턴이 2회 미만, 참고문헌은 18회 이상이라 5.0이면 안전하게 갈립니다.
    """
    density = len(_AUTHOR_INITIALS.findall(chunk)) / max(len(chunk.split()), 1) * 100
    return density >= threshold or chunk.count("[CrossRef]") >= 3

min_token_length = 30

pages_and_chunks = []
n_short = n_refs = 0
for item in tqdm(pages_and_texts, desc="청크 펼치기"):
    for sentence_chunk in item["sentence_chunks"]:
        joined = " ".join(sentence_chunk).strip()
        joined = re.sub(r"\.([A-Z])", r". \1", joined)   # ".Next" -> ". Next"

        if len(joined) / 4 <= min_token_length:
            n_short += 1
            continue
        if looks_like_reference_list(joined):
            n_refs += 1
            continue

        pages_and_chunks.append({
            "pmcid": item["pmcid"],
            "title": item["title"],
            "journal": item["journal"],
            "year": item["year"],
            "page_number": item["page_number"],
            "sentence_chunk": joined,
            "chunk_char_count": len(joined),
            "chunk_word_count": len(joined.split(" ")),
            "chunk_token_count": len(joined) / 4,
        })

print(f"최종 청크: {len(pages_and_chunks)} 개  (짧아서 제외 {n_short}, 참고문헌으로 제외 {n_refs})")
pd.DataFrame(pages_and_chunks)["chunk_token_count"].describe().round(2)

In [ ]:
# 무작위 청크 하나 확인 — 출처 메타데이터가 잘 붙었는지
import random

sample = random.choice(pages_and_chunks)
print(f"출처: {sample['pmcid']} p.{sample['page_number']} — {sample['title']}")
print(f"({sample['journal']} {sample['year']}, {sample['chunk_token_count']:.0f} tokens)\n")
print(sample["sentence_chunk"][:600], "...")

---

## 2단계. 임베딩 만들기 (`all-mpnet-base-v2`, 768차원)

**임베딩**은 텍스트를 의미가 담긴 숫자 벡터로 바꾼 것입니다.
의미가 비슷한 문장은 벡터 공간에서 **가까이** 놓입니다 —
그래서 `"sildenafil mechanism"` 이라는 질의로
`"PDE5 degrades cGMP..."` 라는 문단을 **단어가 겹치지 않아도** 찾아낼 수 있습니다.
(키워드 검색이 못 하는 일이고, 이것이 RAG의 *Retrieval* 입니다.)

`all-mpnet-base-v2` 는 문장 임베딩의 표준 베이스라인으로, 출력이 **768차원**입니다.
성능/속도 비교는 [MTEB 리더보드](https://huggingface.co/spaces/mteb/leaderboard) 참고.

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2",
                                      device=device)

# 임베딩이 무엇인지 감 잡기
demo = embedding_model.encode("Sildenafil inhibits PDE5 and raises cGMP.")
print("임베딩 shape:", demo.shape)      # (768,)
print("앞 8개 값:", demo[:8])

In [ ]:
%%time
# 모든 청크를 임베딩 (batch 처리가 1건씩보다 훨씬 빠릅니다)
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks]

chunk_embeddings = embedding_model.encode(text_chunks,
                                          batch_size=32,
                                          convert_to_tensor=True,
                                          show_progress_bar=True)
print("임베딩 shape:", chunk_embeddings.shape)   # (청크 수, 768)

### CSV로 저장

임베딩 계산은 코퍼스가 커질수록 비싸므로, 한 번 만들고 저장해 재사용합니다.
(실무에서는 CSV 대신 **벡터 DB**(FAISS, Chroma, Qdrant 등)를 씁니다.
청크가 10만 개를 넘어가면 CSV + 완전탐색은 느려집니다.)

In [ ]:
import numpy as np

# 임베딩을 DataFrame에 붙여 저장
embeddings_df_save_path = "pde5_chunks_and_embeddings.csv"

save_df = pd.DataFrame(pages_and_chunks)
save_df["embedding"] = [e.tolist() for e in chunk_embeddings.cpu().numpy()]
save_df.to_csv(embeddings_df_save_path, index=False)

print(f"저장 완료: {embeddings_df_save_path} ({os.path.getsize(embeddings_df_save_path)/1e6:.1f} MB)")
save_df.head(3)

---

## 3단계. 유사도 검색 (dot product top-k)

검색 절차는 세 줄입니다:

1. **질의를 같은 모델로 임베딩** (청크와 같은 벡터 공간에 놓기 위해 — 다른 모델을 쓰면 무의미)
2. 질의 벡터와 모든 청크 벡터의 **내적(dot product)** 계산
3. 점수 상위 **k개** 선택

`all-mpnet-base-v2` 는 출력을 **정규화(normalize)** 해서 내보내므로
내적 = 코사인 유사도이고, 값의 범위는 대략 -1 ~ 1 입니다.

In [ ]:
# CSV에서 다시 불러오기 (여기부터 따로 실행할 수 있게)
import ast

text_chunks_and_embedding_df = pd.read_csv(embeddings_df_save_path)
# CSV에 저장하면 리스트가 문자열이 되므로 되돌립니다
text_chunks_and_embedding_df["embedding"] = (
    text_chunks_and_embedding_df["embedding"].apply(ast.literal_eval)
)

pages_and_chunks = text_chunks_and_embedding_df.to_dict(orient="records")
embeddings = torch.tensor(
    np.array(text_chunks_and_embedding_df["embedding"].tolist()),
    dtype=torch.float32,
).to(device)

print("불러온 임베딩 shape:", embeddings.shape)

In [ ]:
from sentence_transformers import util
from time import perf_counter as timer
import textwrap

def print_wrapped(text, wrap_length=90):
    print(textwrap.fill(text, wrap_length))

def retrieve_relevant_resources(query: str,
                                embeddings: torch.tensor,
                                model: SentenceTransformer = embedding_model,
                                n_resources_to_return: int = 5,
                                print_time: bool = True):
    """질의를 임베딩해 embeddings와의 내적 상위 k개의 (점수, 인덱스)를 반환합니다."""
    query_embedding = model.encode(query, convert_to_tensor=True)

    start_time = timer()
    dot_scores = util.dot_score(query_embedding, embeddings)[0]
    end_time = timer()

    if print_time:
        print(f"[INFO] {len(embeddings)}개 임베딩 검색: {end_time - start_time:.5f}초")

    return torch.topk(input=dot_scores, k=n_resources_to_return)


def print_top_results_and_scores(query: str,
                                 embeddings: torch.tensor,
                                 pages_and_chunks: list[dict] = pages_and_chunks,
                                 n_resources_to_return: int = 5):
    """검색 결과를 출처와 함께 보기 좋게 출력합니다."""
    scores, indices = retrieve_relevant_resources(
        query=query, embeddings=embeddings,
        n_resources_to_return=n_resources_to_return)

    print(f"\nQuery: {query!r}\n" + "=" * 90)
    for score, index in zip(scores, indices):
        item = pages_and_chunks[index]
        print(f"\n[score {score:.4f}] {item['pmcid']} p.{item['page_number']} "
              f"— {item['title']} ({item['journal']} {item['year']})")
        print_wrapped(item["sentence_chunk"][:700])

### 질의 1 — 작용기전

원본 튜토리얼의 영양학 질의(`"macronutrients functions"` 등)를 **PDE5 도메인 질의**로 바꿉니다.

> **중요 — RAG에서 가장 흔한 실패:** 코퍼스는 그대로 두고 질의만 바꾸면,
> 검색은 "그나마 비슷해 보이는 아무 문단"을 가져오고 LLM은 그 위에 그럴듯한 답을 **지어냅니다**.
> 질의와 코퍼스는 **항상 같이** 바꿔야 합니다. (이 노트북은 둘 다 PDE5로 교체했습니다.)

In [ ]:
query = "mechanism of sildenafil as a PDE5 inhibitor"
print_top_results_and_scores(query=query, embeddings=embeddings, n_resources_to_return=3)

### 질의 2·3 — 안전성과 리포지셔닝

질의를 바꾸면 **검색이 다른 논문을 집어오는지** 확인합니다.
이것이 코퍼스를 세 도메인(리포지셔닝 / 발기부전 / 심혈관 안전성)으로 나눠 고른 이유입니다.

In [ ]:
pde5_queries = [
    "mechanism of sildenafil as a PDE5 inhibitor",
    "cardiovascular safety and arrhythmia risk of PDE5 inhibitors",
    "PDE5 inhibitor repurposing beyond erectile dysfunction",
    "sildenafil effect on angiogenesis and collateral vessel growth",
    "PDE5 inhibitors and cerebral small vessel disease blood flow",
]

for q in pde5_queries[1:3]:
    print_top_results_and_scores(query=q, embeddings=embeddings, n_resources_to_return=3)
    print("\n")

In [ ]:
# 어떤 질의가 어떤 논문을 끌어오는지 한눈에 보기
rows = []
for q in pde5_queries:
    scores, indices = retrieve_relevant_resources(q, embeddings,
                                                  n_resources_to_return=5,
                                                  print_time=False)
    hits = [pages_and_chunks[i]["pmcid"] for i in indices]
    rows.append({"query": q[:52],
                 "top1": hits[0],
                 "top5 논문 분포": ", ".join(f"{p}×{hits.count(p)}"
                                            for p in dict.fromkeys(hits))})
pd.DataFrame(rows)

---

## 4단계. 로컬 LLM 준비

검색까지는 LLM이 필요 없었습니다. 이제 **찾은 문맥을 근거로 답을 쓰는** 단계입니다.

- 기본값 **`Qwen/Qwen3-1.7B`** — Apache-2.0, **gated 아님(HF 로그인 불필요)**, CPU에서도 동작
- GPU 메모리가 넉넉하면 `Qwen/Qwen3-8B` 로 자동 상향
- `google/gemma-2b-it` 은 **gated** 라 라이선스 동의 + 토큰이 필요합니다 → **선택 사항**

In [ ]:
# 사용 가능한 GPU 메모리 확인 (CPU 런타임이면 0)
if torch.cuda.is_available():
    gpu_memory_gb = round(torch.cuda.get_device_properties(0).total_memory / (2**30))
    print(f"Available GPU memory: {gpu_memory_gb} GB ({torch.cuda.get_device_name(0)})")
else:
    gpu_memory_gb = 0
    print("[안내] GPU가 없습니다. 생성 LLM은 CPU에서 돌아가며 답변 1건에 1~3분 걸릴 수 있습니다.")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device)

In [ ]:
# 사용할 로컬 LLM 선택
#
# 원본 튜토리얼은 Gemma를 사용했지만, Gemma는 Hugging Face에서 "수동 승인"이 필요한
# gated 모델이라 로그인 없이 from_pretrained를 호출하면 401 오류가 납니다.
# 기본값은 승인이 필요 없는 Qwen3로 두고, Gemma는 선택 사항으로 남겨둡니다.

USE_GATED_GEMMA = False  # True로 바꾸면 Gemma를 사용합니다 (아래 로그인 필요)

# 4bit 양자화(bitsandbytes)는 NVIDIA GPU 전용입니다. CPU 런타임에서는 항상 끕니다.
CAN_QUANTIZE = torch.cuda.is_available()

if USE_GATED_GEMMA:
    # 1) https://huggingface.co/google/gemma-2b-it 에서 라이선스 동의
    # 2) https://huggingface.co/settings/tokens 에서 토큰 발급
    from huggingface_hub import login
    from getpass import getpass
    login(token=getpass("Hugging Face 토큰을 입력하세요: "))

    model_id = "google/gemma-2b-it" if gpu_memory_gb < 19.0 else "google/gemma-7b-it"
    use_quantization_config = CAN_QUANTIZE and gpu_memory_gb < 8.1
    if gpu_memory_gb and gpu_memory_gb < 5.1:
        print(f"GPU memory {gpu_memory_gb}GB: 양자화 없이는 Gemma 실행이 어려울 수 있습니다.")
else:
    # 승인이 필요 없는 Qwen3 계열
    model_id = "Qwen/Qwen3-8B" if gpu_memory_gb >= 19.0 else "Qwen/Qwen3-1.7B"
    use_quantization_config = CAN_QUANTIZE and gpu_memory_gb < 8.1

# Qwen3는 thinking 모드를 지원합니다. RAG 답변에 <think> 블록이 섞이지 않도록
# 뒤쪽 apply_chat_template 호출에서 enable_thinking=False 를 넘깁니다.
IS_QWEN3 = model_id.startswith("Qwen/Qwen3")

print(f"use_quantization_config set to: {use_quantization_config}")
print(f"model_id set to: {model_id}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import is_flash_attn_2_available

# 1. 4bit 양자화 설정 (GPU 전용, 선택)
quantization_config = None
if use_quantization_config:
    from transformers import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(load_in_4bit=True,
                                             bnb_4bit_compute_dtype=torch.float16)

# 2. Flash Attention 2 는 compute capability 8.0 이상 NVIDIA GPU + flash-attn 설치 필요.
#    없으면 "sdpa"(scaled dot product attention)로 자동 fallback.
if (torch.cuda.is_available() and is_flash_attn_2_available()
        and torch.cuda.get_device_capability(0)[0] >= 8):
    attn_implementation = "flash_attention_2"
else:
    attn_implementation = "sdpa"
print(f"[INFO] Using attention implementation: {attn_implementation}")
print(f"[INFO] Using model_id: {model_id}")

# 3. 토크나이저
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_id)

# 4. 모델
#    CPU 에서는 float16 연산이 매우 느리거나 미지원이라 float32 를 씁니다.
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

#    transformers 4.56 부터 인자 이름이 torch_dtype -> dtype 로 바뀌었습니다(구 이름은 deprecated).
#    Colab의 transformers 버전이 무엇이든 동작하도록 이름을 골라서 넘깁니다.
import transformers
from packaging.version import parse as _v
_dtype_kw = "dtype" if _v(transformers.__version__) >= _v("4.56") else "torch_dtype"
print(f"[INFO] transformers {transformers.__version__} -> using '{_dtype_kw}' kwarg")

llm_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_id,
    quantization_config=quantization_config,
    low_cpu_mem_usage=False,
    attn_implementation=attn_implementation,
    **{_dtype_kw: model_dtype},
)

if not use_quantization_config:  # 양자화를 쓰면 device 배치가 자동으로 됩니다
    llm_model.to(device)

llm_model.eval()

def get_model_num_params(model): return sum(p.numel() for p in model.parameters())
print(f"[INFO] LLM loaded on: {next(llm_model.parameters()).device}")
print(f"[INFO] 파라미터 수: {get_model_num_params(llm_model)/1e9:.2f}B")

---

## 5단계. 프롬프트 증강(Augmentation) → 생성(Generation)

RAG의 **A**(Augmented)와 **G**(Generation)입니다. 검색된 청크를 프롬프트에 끼워 넣고,
**"이 문맥만 근거로 답하라"** 고 지시합니다.

### 원본과 달라진 점 — few-shot 예시를 뺐습니다

원본 튜토리얼은 프롬프트에 영양학 모범답안 3개(지용성 비타민, 제2형 당뇨병, 수분 섭취)를
few-shot 예시로 넣었습니다. 이를 PDE5 버전으로 바꾸려면 **제가 PDE5 사실관계를 직접 써넣어야
하는데**, 그러면 코퍼스에 없는 내용이 프롬프트를 통해 답변에 새어 들어갑니다
(모델이 예시 문장을 그대로 베끼는 일이 실제로 자주 일어납니다).

RAG 실습의 목적은 **답이 검색된 문맥에서 나오는지 확인**하는 것이므로,
모범답안 대신 **답변 형식 규칙**만 지시하고 **출처(PMCID·쪽)를 반드시 밝히도록** 했습니다.

### `apply_chat_template`

instruction-tuned 모델은 `<|im_start|>user ... <|im_end|>` 같은 **고유한 대화 서식**으로
학습됩니다. 이 서식을 안 맞추면 품질이 눈에 띄게 떨어지므로,
직접 문자열을 조립하지 말고 토크나이저의 `apply_chat_template` 을 씁니다.

In [ ]:
def prompt_formatter(query: str, context_items: list[dict]) -> str:
    """검색된 문맥으로 질의를 증강해 모델용 프롬프트를 만듭니다."""
    # 문맥마다 출처를 붙여, 모델이 인용할 수 있게 합니다
    context = "\n\n".join(
        f"[{item['pmcid']} p.{item['page_number']}] {item['sentence_chunk']}"
        for item in context_items
    )

    base_prompt = """You are a biomedical research assistant. Answer the user's query
using ONLY the context passages below, which are excerpts from open-access papers on
PDE5 (phosphodiesterase type 5) inhibitors.

Rules:
- Base every factual statement on the context passages. Do not add outside knowledge.
- If the context does not contain the answer, say so explicitly instead of guessing.
- Cite the source of each claim inline using its tag, e.g. [PMC13149040 p.4].
- Be explanatory and specific; prefer mechanisms and numbers found in the context.
- Return only the final answer, not your reasoning.

Context passages:
{context}

User query: {query}
Answer:"""

    base_prompt = base_prompt.format(context=context, query=query)

    dialogue_template = [{"role": "user", "content": base_prompt}]

    # instruction-tuned 모델의 고유 대화 서식을 적용
    return tokenizer.apply_chat_template(
        conversation=dialogue_template,
        tokenize=False,
        add_generation_prompt=True,
        # Qwen3의 thinking 모드를 꺼서 <think> 블록을 제거
        **({"enable_thinking": False} if IS_QWEN3 else {}),
    )

In [ ]:
# 프롬프트가 어떻게 생겼는지 확인
_q = "mechanism of sildenafil as a PDE5 inhibitor"
_scores, _indices = retrieve_relevant_resources(_q, embeddings, n_resources_to_return=3)
_ctx = [pages_and_chunks[i] for i in _indices]

_prompt = prompt_formatter(query=_q, context_items=_ctx)
print(_prompt[:1800], "\n...\n")
print("프롬프트 토큰 수:", len(tokenizer(_prompt)["input_ids"]))

### `ask()` — 검색 + 증강 + 생성을 한 번에

파이프라인 전체를 함수 하나로 묶습니다.

> **CPU 안내:** CPU에서는 초당 3토큰 안팎이라 `max_new_tokens=256` 이면 1~2분 걸립니다.
> GPU가 있으면 512로 늘려도 몇 초면 끝납니다.

In [ ]:
def ask(query: str,
        temperature: float = 0.7,
        max_new_tokens: int = 256,
        n_resources_to_return: int = 5,
        return_context: bool = False):
    """질의 -> 관련 문맥 검색 -> 프롬프트 증강 -> LLM 답변 생성."""
    scores, indices = retrieve_relevant_resources(
        query=query, embeddings=embeddings,
        n_resources_to_return=n_resources_to_return)

    context_items = [pages_and_chunks[i] for i in indices]
    for i, item in enumerate(context_items):
        item["score"] = scores[i].cpu().item()

    prompt = prompt_formatter(query=query, context_items=context_items)
    input_ids = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = llm_model.generate(**input_ids,
                                 temperature=temperature,
                                 do_sample=True,
                                 max_new_tokens=max_new_tokens)

    # 새로 생성된 토큰만 잘라 디코딩 (문자열 replace보다 안전)
    generated = outputs[0][input_ids["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()

    if return_context:
        return answer, context_items
    return answer

In [ ]:
%%time
query = "mechanism of sildenafil as a PDE5 inhibitor"
answer, context_items = ask(query=query, max_new_tokens=256, return_context=True)

print(f"Query: {query}\n" + "=" * 90)
print("\n[Answer]")
print_wrapped(answer)

print("\n" + "=" * 90)
print("[근거로 사용된 문맥]")
for item in context_items:
    print(f"  - score {item['score']:.3f}  {item['pmcid']} p.{item['page_number']} — {item['title']}")

### 직접 해보기

`pde5_queries` 의 다른 질의로도 돌려보세요. 확인할 것:

1. **답변이 검색된 문맥에 실제로 있는 내용인가?** (출처 태그를 따라가 원문과 대조)
2. **질의를 바꾸면 검색 결과가 정말 바뀌는가?**
3. **코퍼스에 없는 질문을 던지면?** — 예: `"What is the half-life of aspirin?"`
   모델이 "문맥에 없다"고 말하는지, 아니면 지어내는지 보세요.
   이것이 RAG 시스템 평가의 출발점입니다.

In [ ]:
# 코퍼스 범위 밖 질문 — 모델이 "모른다"고 하는지 확인
off_topic = "What is the elimination half-life of aspirin in humans?"
print(f"Query: {off_topic}\n" + "=" * 90)
print_wrapped(ask(query=off_topic, max_new_tokens=160))

---

## 정리

우리가 처음부터 만든 것:

| 단계 | 결과물 |
|---|---|
| 1. 추출 | PDE5 논문 PDF 6편 → 쪽 단위 텍스트 (출처 메타데이터 포함) |
| 2. 청킹 | spaCy 문장 분리 → 10문장 청크 → 짧은 조각·참고문헌 제거 |
| 3. 임베딩 | `all-mpnet-base-v2` 768차원 → CSV 저장 |
| 4. 검색 | 질의 임베딩 → dot product → top-k |
| 5. 생성 | 문맥 증강 프롬프트 → `apply_chat_template` → Qwen3 답변 + 출처 인용 |

LangChain이나 LlamaIndex를 쓰면 이 전부가 몇 줄로 줄어듭니다.
직접 만들어 본 이유는 **각 단계에서 무엇이 잘못될 수 있는지** 보기 위해서입니다:

- 청크가 너무 크면/작으면 검색이 무너진다
- 참고문헌 같은 노이즈가 상위를 차지한다 (그래서 필터를 넣었습니다)
- 질의와 코퍼스의 도메인이 어긋나면 **그럴듯한 거짓말**이 나온다
- 임베딩 모델과 질의 임베딩 모델이 다르면 검색 자체가 무의미하다

### 한계 (이 노트북 기준)

- 청크 6편·수백 개 규모라 완전탐색으로 충분하지만, 10만 개를 넘으면 **벡터 DB**가 필요합니다.
- 검색 품질을 **정량 평가하지 않았습니다** (recall@k, MRR 등). 실무에서는 필수입니다.
- 참고문헌 필터는 휴리스틱이라 완벽하지 않습니다 — 마지막 쪽의
  "Declaration of competing interest + References" 혼합 청크는 여전히 걸러지지 않습니다.

## 더 알아보기

지면상 이 노트북에서 **덜어낸** 주제들입니다. 모두 원저자
**Daniel Bourke** 의 원본 노트북
([`mrdbourke/simple-local-rag`](https://github.com/mrdbourke/simple-local-rag))
에 자세히 설명되어 있으니, 더 파고들고 싶으면 원본을 보세요.

**텍스트 추출 고도화**
- [Marker](https://github.com/VikParuchuri/marker) — PDF → Markdown 고품질 변환
- 표·그림까지 다루려면 레이아웃 인식 파서가 필요합니다

**검색 고도화**
- 벡터 DB / 인덱스: [FAISS](https://github.com/facebookresearch/faiss), Chroma, Qdrant
- 다른 임베딩 모델: [`mixedbread-ai/mxbai-embed-large-v1`](https://huggingface.co/mixedbread-ai/mxbai-embed-large-v1),
  [MTEB 리더보드](https://huggingface.co/spaces/mteb/leaderboard)
- Re-ranking (cross-encoder), hybrid search (BM25 + 벡터)

**생성 고도화 / 추론 최적화**
- [Hugging Face GPU 추론 최적화 가이드](https://huggingface.co/docs/transformers/perf_infer_gpu_one)
- [Flash Attention 2](https://github.com/Dao-AILab/flash-attention) (Ampere 이상 GPU)
- [Optimum NVIDIA](https://github.com/huggingface/optimum-nvidia),
  [TensorRT-LLM](https://github.com/NVIDIA/TensorRT-LLM),
  [GPT-Fast](https://github.com/pytorch-labs/gpt-fast)
- 토큰 스트리밍 출력, 다른 LLM (Mistral-Instruct 등)

**평가와 응용**
- LLM-as-a-judge 로 답변 채점, recall@k / MRR 로 검색 평가
- 프레임워크: [LangChain](https://www.langchain.com/), [LlamaIndex](https://www.llamaindex.ai/)
- 앱으로 만들기: [Gradio 챗봇 가이드](https://www.gradio.app/guides/creating-a-chatbot-fast)

---

*원본 튜토리얼 © Daniel Bourke ([mrdbourke/simple-local-rag](https://github.com/mrdbourke/simple-local-rag)).
본 노트북은 해당 튜토리얼을 기반으로 PDE5 신약개발 도메인에 맞춰 개편한 비영리 교육용 자료입니다.
원본 저장소에 명시된 라이선스가 없으므로, 재배포 전 원저자의 허락이 필요합니다.*